In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import pandas as pd
from collections import OrderedDict

from single_field_features import extract_single_field_features
from pairwise_field_features import extract_pairwise_field_features
from aggregate_single_field_features import extract_aggregate_single_field_features
from aggregate_pairwise_field_features import extract_aggregate_pairwise_field_features

compute_features_config = {
    'single_field': True,
    'aggregate_single_field': True,

    'pairwise_field': True,
    'aggregate_pairwise_field': True,

    'field_level_features': False,
    'chart_outcomes': False,
    'field_outcomes': False,
    'supplement': False,
}

def extract_features_from_fields(fields, compute_features_config, fid=None):
    """
    参考你给出的代码逻辑，去掉 clean_chunk / chart_outcomes 等不需要的部分。
    fields: 形如 [ (col_name, {'uid': ..., 'order': ..., 'data': ...}), (...), ... ]
    fid: 一个字符串，用来标识这个数据集或表格。
    """
    results = {}
    
    MAX_FIELDS = len(fields)

    df_feature_tuples = OrderedDict({'fid': fid})
    df_feature_tuples_if_exists = OrderedDict({'fid': fid})

    single_field_features, parsed_fields = ([], [])
    if compute_features_config['single_field'] or compute_features_config['field_level_features']:
        single_field_features, parsed_fields = extract_single_field_features(
            fields,
            fid,
            MAX_FIELDS=MAX_FIELDS
        )
        for i, field_features in enumerate(single_field_features):
            if not field_features.get('exists'):
                continue
            field_num = i + 1
            for field_feature_name, field_feature_value in field_features.items():
                if field_feature_name not in ['fid', 'field_id', 'exists']:
                    field_feature_name_with_num = f"{field_feature_name}_{field_num}"
                    df_feature_tuples_if_exists[field_feature_name_with_num] = field_feature_value
        
        results['single_field_features'] = single_field_features

    if compute_features_config['aggregate_single_field']:
        aggregate_single_field_features = extract_aggregate_single_field_features(
            single_field_features
        )
        for k, v in aggregate_single_field_features.items():
            df_feature_tuples[k] = v
            df_feature_tuples_if_exists[k] = v
        results['aggregate_single_field_features'] = aggregate_single_field_features

    pairwise_field_features_result = []
    if compute_features_config['pairwise_field'] or compute_features_config['aggregate_pairwise_field']:
        pairwise_field_features_result = extract_pairwise_field_features(
            parsed_fields,
            single_field_features,
            fid,
            MAX_FIELDS=MAX_FIELDS
        )
        results['pairwise_field_features'] = pairwise_field_features_result

    if compute_features_config['aggregate_pairwise_field']:
        aggregate_pairwise_field_features = extract_aggregate_pairwise_field_features(
            pairwise_field_features_result
        )
        for k, v in aggregate_pairwise_field_features.items():
            df_feature_tuples[k] = v
            df_feature_tuples_if_exists[k] = v
        results['aggregate_pairwise_field_features'] = aggregate_pairwise_field_features

    # if compute_features_config['chart_outcomes']:
    #     ...
    # if compute_features_config['field_outcomes']:
    #     ...

    results['df_feature_tuples'] = df_feature_tuples
    results['df_feature_tuples_if_exists'] = df_feature_tuples_if_exists

    return results


def main():
    df = pd.read_csv('Books.csv')

    fields = []
    for idx, col_name in enumerate(df.columns):
        col_data = df[col_name].tolist()
        field_info = {
            'uid': col_name,
            'order': col_name,
            'data': col_data
        }
        fields.append((col_name, field_info))
    
    fid = "seattle_weather_dataset"

    extraction_results = extract_features_from_fields(
        fields=fields,
        compute_features_config=compute_features_config,
        fid=fid
    )

    single_field_features = extraction_results.get('single_field_features', [])
    pairwise_field_features = extraction_results.get('pairwise_field_features', [])
    aggregated_single_field_features = extraction_results.get('aggregate_single_field_features', {})
    aggregated_pairwise_field_features = extraction_results.get('aggregate_pairwise_field_features', {})

    print(single_field_features)
    df_single = pd.DataFrame(single_field_features)
    print("Shape of df_single:", df_single.shape)
    df_single.to_csv("single_field_features.csv", index=False)

    flattened_pairwise = []
    for row in pairwise_field_features:
        flattened_pairwise.extend(row)
    df_pairwise = pd.DataFrame(flattened_pairwise)
    print("Shape of df_pairwise:", df_pairwise.shape)
    df_pairwise.to_csv("pairwise_field_features.csv", index=False)

    df_agg_single = pd.DataFrame([aggregated_single_field_features])
    print("Shape of df_agg_single:", df_agg_single.shape)
    df_agg_single.to_csv("aggregated_single_field_features.csv", index=False)

    df_agg_pairwise = pd.DataFrame([aggregated_pairwise_field_features])
    print("Shape of df_agg_pairwise:", df_agg_pairwise.shape)
    df_agg_pairwise.to_csv("aggregated_pairwise_field_features.csv", index=False)

    print("特征提取完成，已生成以下4个文件：")
    print("  - single_field_features.csv")
    print("  - pairwise_field_features.csv")
    print("  - aggregated_single_field_features.csv")
    print("  - aggregated_pairwise_field_features.csv")



if __name__ == "__main__":
    main()


[OrderedDict([('fid', 'seattle_weather_dataset'), ('field_id', 'seattle_weather_dataset:Rank'), ('exists', True), ('length', 100), ('data_type_is_string', False), ('data_type_is_integer', True), ('data_type_is_decimal', False), ('data_type_is_time', False), ('general_type_is_c', False), ('general_type_is_q', True), ('general_type_is_t', False), ('has_none', False), ('percentage_none', 0.0), ('num_none', 0), ('num_unique_elements', 100), ('unique_percent', 1.0), ('is_unique', True), ('list_entropy', None), ('mean_value_length', None), ('median_value_length', None), ('min_value_length', None), ('max_value_length', None), ('std_value_length', None), ('percentage_of_mode', None), ('mean', np.float64(50.5)), ('normalized_mean', np.float64(0.505)), ('median', np.float64(50.5)), ('normalized_median', np.float64(0.505)), ('var', np.float64(833.25)), ('std', np.float64(28.86607004772212)), ('coeff_var', np.float64(16.5)), ('min', np.int64(1)), ('max', np.int64(100)), ('range', np.int64(99)), ('

/Users/fred/contribution2/vizml-master/feature_extraction/features/helpers.py:102: RuntimeWarning: invalid value encountered in cast
  return v.astype(np.integer)
/Users/fred/contribution2/vizml-master/feature_extraction/features/single_field_features.py:214: RuntimeWarning: invalid value encountered in scalar divide
  r['quant_coeff_disp'] = (q75 - q25) / (q75 + q25)
/Users/fred/contribution2/vizml-master/feature_extraction/features/single_field_features.py:306: RuntimeWarning: invalid value encountered in divide
  sequence_incremental_division = np.divide(sorted_v[:-1], sorted_v[1:])


In [1]:
def main():
    df = pd.read_csv('seattle_weather.csv')

    fields = []
    for idx, col_name in enumerate(df.columns):
        col_data = df[col_name].tolist()
        field_info = {
            'uid': col_name,
            'order': col_name,
            'data': col_data
        }
        fields.append((col_name, field_info))
    
    fid = "seattle_weather_dataset"

    extraction_results = extract_features_from_fields(
        fields=fields,
        compute_features_config=compute_features_config,
        fid=fid
    )

    single_field_features = extraction_results.get('single_field_features', [])
    pairwise_field_features = extraction_results.get('pairwise_field_features', [])
    aggregated_single_field_features = extraction_results.get('aggregate_single_field_features', {})
    aggregated_pairwise_field_features = extraction_results.get('aggregate_pairwise_field_features', {})

    df_single = pd.DataFrame(single_field_features)
    df_single.drop(['fid', 'field_id', 'exists'], axis=1, inplace=True, errors='ignore')
    df_single.to_csv("single_field_features.csv", index=False)
    print("Shape of df_single (仅保留特征值):", df_single.shape)

    flattened_pairwise = []
    for row in pairwise_field_features:
        flattened_pairwise.extend(row)
    df_pairwise = pd.DataFrame(flattened_pairwise)
    df_pairwise.drop(['fid', 'field_a_id', 'field_b_id'], axis=1, inplace=True, errors='ignore')
    df_pairwise.to_csv("pairwise_field_features.csv", index=False)
    print("Shape of df_pairwise (仅保留特征值):", df_pairwise.shape)

    df_agg_single = pd.DataFrame([aggregated_single_field_features])
    df_agg_single.drop(['fid'], axis=1, inplace=True, errors='ignore')
    df_agg_single.to_csv("aggregated_single_field_features.csv", index=False)
    print("Shape of df_agg_single (仅保留特征值):", df_agg_single.shape)

    df_agg_pairwise = pd.DataFrame([aggregated_pairwise_field_features])
    df_agg_pairwise.drop(['fid'], axis=1, inplace=True, errors='ignore')
    df_agg_pairwise.to_csv("aggregated_pairwise_field_features.csv", index=False)
    print("Shape of df_agg_pairwise (仅保留特征值):", df_agg_pairwise.shape)

    print("特征提取完成，已生成以下4个只含特征值的文件：")
    print("  - single_field_features.csv")
    print("  - pairwise_field_features.csv")
    print("  - aggregated_single_field_features.csv")
    print("  - aggregated_pairwise_field_features.csv")


if __name__ == "__main__":
    main()

NameError: name 'pd' is not defined

In [8]:
# Import necessary libraries
import pandas as pd
from collections import OrderedDict

# Import your feature extraction functions
from single_field_features import extract_single_field_features
from pairwise_field_features import extract_pairwise_field_features
from aggregate_single_field_features import extract_aggregate_single_field_features
from aggregate_pairwise_field_features import extract_aggregate_pairwise_field_features

# Load your dataset
df = pd.read_csv('Books.csv')

# Prepare fields for feature extraction
fields = []
for idx, col_name in enumerate(df.columns):
    col_data = df[col_name].tolist()
    field_info = {
        'uid': col_name,
        'order': col_name,
        'data': col_data
    }
    fields.append((col_name, field_info))

# Set the feature computation configuration
compute_features_config = {
    'single_field': True,
    'aggregate_single_field': True,
    'pairwise_field': True,
    'aggregate_pairwise_field': True,
    'field_level_features': False,
    'chart_outcomes': False,
    'field_outcomes': False,
    'supplement': False,
}

# Extract features
extraction_results = extract_features_from_fields(
    fields=fields,
    compute_features_config=compute_features_config,
    fid='seattle_weather_dataset'
)

# Get the features
single_field_features = extraction_results.get('single_field_features', [])
pairwise_field_features = extraction_results.get('pairwise_field_features', [])
aggregated_single_field_features = extraction_results.get('aggregate_single_field_features', {})
aggregated_pairwise_field_features = extraction_results.get('aggregate_pairwise_field_features', {})

# Convert OrderedDicts to DataFrames
df_agg_single = pd.DataFrame([aggregated_single_field_features])
df_agg_pairwise = pd.DataFrame([aggregated_pairwise_field_features])

# Print the shapes of the features
print("Shape of single_field_features:", len(single_field_features), "x", len(single_field_features[0]) if single_field_features else 0)
# Print the shapes of the features
print("Shape of single_field_features:", len(single_field_features), "x", len(single_field_features[0]) if single_field_features else 0)
print("Shape of pairwise_field_features:", len(pairwise_field_features), "x", len(pairwise_field_features[0]) if pairwise_field_features else 0)
print("Shape of aggregated_single_field_features:", 1, "x", len(aggregated_single_field_features))
print("Shape of aggregated_pairwise_field_features:", 1, "x", len(aggregated_pairwise_field_features))

# Combine the columns of both DataFrames
all_features = set(df_agg_single.columns).union(set(df_agg_pairwise.columns))

# Calculate the total unique features
total_features = len(all_features)
print(f"Total unique features: {total_features}")
overlapping_features = set(df_agg_single.columns).intersection(set(df_agg_pairwise.columns))
print(f"Number of overlapping features: {len(overlapping_features)}")
# Normalize column names
df_agg_single.columns = df_agg_single.columns.str.strip().str.lower()
df_agg_pairwise.columns = df_agg_pairwise.columns.str.strip().str.lower()

# Recalculate total unique features
all_features_normalized = set(df_agg_single.columns).union(set(df_agg_pairwise.columns))
print(f"Total unique features after normalization: {len(all_features_normalized)}")

Shape of single_field_features: 8 x 83
Shape of single_field_features: 8 x 83
Shape of pairwise_field_features: 7 x 7
Shape of aggregated_single_field_features: 1 x 657
Shape of aggregated_pairwise_field_features: 1 x 240
Total unique features: 897
Number of overlapping features: 0
Total unique features after normalization: 897


/Users/fred/contribution2/vizml-master/feature_extraction/features/helpers.py:102: RuntimeWarning: invalid value encountered in cast
  return v.astype(np.integer)
/Users/fred/contribution2/vizml-master/feature_extraction/features/single_field_features.py:214: RuntimeWarning: invalid value encountered in scalar divide
  r['quant_coeff_disp'] = (q75 - q25) / (q75 + q25)
/Users/fred/contribution2/vizml-master/feature_extraction/features/single_field_features.py:306: RuntimeWarning: invalid value encountered in divide
  sequence_incremental_division = np.divide(sorted_v[:-1], sorted_v[1:])


In [18]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import pandas as pd
from collections import OrderedDict

from single_field_features import extract_single_field_features
from pairwise_field_features import extract_pairwise_field_features
from aggregate_single_field_features import extract_aggregate_single_field_features
from aggregate_pairwise_field_features import extract_aggregate_pairwise_field_features

compute_features_config = {
    'single_field': True,
    'aggregate_single_field': True,
    'pairwise_field': True,
    'aggregate_pairwise_field': True,
    'field_level_features': False,
    'chart_outcomes': False,
    'field_outcomes': False,
    'supplement': False,
}

def extract_features_from_fields(fields, compute_features_config, fid=None):
    """
    提取特征的主函数
    """
    results = {}
    
    MAX_FIELDS = len(fields)
    
    df_feature_tuples = OrderedDict({'fid': fid})
    df_feature_tuples_if_exists = OrderedDict({'fid': fid})

    single_field_features, parsed_fields = ([], [])
    if compute_features_config['single_field'] or compute_features_config['field_level_features']:
        single_field_features, parsed_fields = extract_single_field_features(
            fields, fid, MAX_FIELDS=MAX_FIELDS
        )
        for i, field_features in enumerate(single_field_features):
            if not field_features.get('exists'):
                continue
            field_num = i + 1
            for field_feature_name, field_feature_value in field_features.items():
                if field_feature_name not in ['fid', 'field_id', 'exists']:
                    field_feature_name_with_num = f"{field_feature_name}_{field_num}"
                    df_feature_tuples_if_exists[field_feature_name_with_num] = field_feature_value
        results['single_field_features'] = single_field_features

    if compute_features_config['aggregate_single_field']:
        aggregate_single_field_features = extract_aggregate_single_field_features(single_field_features)
        for k, v in aggregate_single_field_features.items():
            df_feature_tuples[k] = v
            df_feature_tuples_if_exists[k] = v
        results['aggregate_single_field_features'] = aggregate_single_field_features

    pairwise_field_features_result = []
    if compute_features_config['pairwise_field'] or compute_features_config['aggregate_pairwise_field']:
        pairwise_field_features_result = extract_pairwise_field_features(
            parsed_fields, single_field_features, fid, MAX_FIELDS=MAX_FIELDS
        )
        
        for pairwise_feature in pairwise_field_features_result:
            for feature in pairwise_feature:
                feature['fid'] = fid
                feature['field_a_id'] = feature.get('field_a_name', '')
                feature['field_b_id'] = feature.get('field_b_name', '')
        
        results['pairwise_field_features'] = pairwise_field_features_result

    if compute_features_config['aggregate_pairwise_field']:
        aggregate_pairwise_field_features = extract_aggregate_pairwise_field_features(pairwise_field_features_result)
        for k, v in aggregate_pairwise_field_features.items():
            df_feature_tuples[k] = v
            df_feature_tuples_if_exists[k] = v
        results['aggregate_pairwise_field_features'] = aggregate_pairwise_field_features

    results['df_feature_tuples'] = df_feature_tuples
    results['df_feature_tuples_if_exists'] = df_feature_tuples_if_exists

    return results

def main():
    df = pd.read_csv('seattle_weather.csv')

    fields = []
    for idx, col_name in enumerate(df.columns):
        col_data = df[col_name].tolist()
        field_info = {
            'uid': col_name,
            'order': col_name,
            'data': col_data
        }
        fields.append((col_name, field_info))
    
    fid = "seattle_weather_dataset"

    extraction_results = extract_features_from_fields(fields=fields, compute_features_config=compute_features_config, fid=fid)

    single_field_features = extraction_results.get('single_field_features', [])
    pairwise_field_features = extraction_results.get('pairwise_field_features', [])
    aggregated_single_field_features = extraction_results.get('aggregate_single_field_features', {})
    aggregated_pairwise_field_features = extraction_results.get('aggregate_pairwise_field_features', {})

    df_single = pd.DataFrame(single_field_features)
    df_single.to_csv("single_field_features.csv", index=False)

    flattened_pairwise = []
    for row in pairwise_field_features:
        flattened_pairwise.extend(row)
    df_pairwise = pd.DataFrame(flattened_pairwise)
    df_pairwise.to_csv("pairwise_field_features.csv", index=False)

    df_agg_single = pd.DataFrame([aggregated_single_field_features])
    df_agg_single.to_csv("aggregated_single_field_features.csv", index=False)

    df_agg_pairwise = pd.DataFrame([aggregated_pairwise_field_features])
    df_agg_pairwise.to_csv("aggregated_pairwise_field_features.csv", index=False)

    print("特征提取完成，已生成以下4个文件：")
    print("  - single_field_features.csv")
    print("  - pairwise_field_features.csv")
    print("  - aggregated_single_field_features.csv")
    print("  - aggregated_pairwise_field_features.csv")

if __name__ == "__main__":
    main()


int() argument must be a string, a bytes-like object or a number, not 'Timestamp'
Error parsing date: int() argument must be a string, a bytes-like object or a number, not 'Timestamp'
特征提取完成，已生成以下4个文件：
  - single_field_features.csv
  - pairwise_field_features.csv
  - aggregated_single_field_features.csv
  - aggregated_pairwise_field_features.csv


/Users/fred/contribution2/vizml-master/feature_extraction/features/helpers.py:116: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(
/Users/fred/contribution2/vizml-master/feature_extraction/features/single_field_features.py:214: RuntimeWarning: invalid value encountered in scalar divide
  r['quant_coeff_disp'] = (q75 - q25) / (q75 + q25)
/Users/fred/contribution2/vizml-master/feature_extraction/features/single_field_features.py:306: RuntimeWarning: invalid value encountered in divide
  sequence_incremental_division = np.divide(sorted_v[:-1], sorted_v[1:])
/Users/fred/contribution2/vizml-master/feature_extraction/features/single_field_features.py:306: RuntimeWarning: divide by zero encountered in divide
  sequence_incremental_division = np.divide(sorte